In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [3]:
# 1. 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/mobile.csv")

print(df.head())
print(df.shape)

   battery_power  blue  clock_speed  dual_sim  fc  four_g  int_memory  m_dep  \
0            842     0          2.2         0   1       0           7    0.6   
1           1021     1          0.5         1   0       1          53    0.7   
2            563     1          0.5         1   2       1          41    0.9   
3            615     1          2.5         0   0       0          10    0.8   
4           1821     1          1.2         0  13       1          44    0.6   

   mobile_wt  n_cores  ...  px_height  px_width   ram  sc_h  sc_w  talk_time  \
0        188        2  ...         20       756  2549     9     7         19   
1        136        3  ...        905      1988  2631    17     3          7   
2        145        5  ...       1263      1716  2603    11     2          9   
3        131        6  ...       1216      1786  2769    16     8         11   
4        141        2  ...       1208      1212  1411     8     2         15   

   three_g  touch_screen  wifi  price_

In [4]:
# 2. 입력/정답 분리
X = df.drop(["price_range", "three_g", "battery_power"], axis=1).values
y = df["battery_power"].values

# 회귀는 연속적인 숫자를 예측하므로 y값에 원-핫 인코딩을 적용하지 않음

In [5]:
# 3. 학습 / 테스트 데이터 분리
# y가 연속형 수치 데이터이므로 stratify=y 제거
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [6]:
# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [7]:
# 5. 모델 생성
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation="relu"))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(1))
# X_train.shape[1]을 사용하면 입력 특성 개수가 바껴도 수정 필요 없음
# 회귀이므로 출력층 활성화함수 제거

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [8]:
# 6. 모델 설정
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
# 회귀이므로 손실함수 mse, 매트릭스 mae 사용

In [9]:
# 7. 모델 구조 출력
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 64)             │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,841 (15.00 KB)

 Trainable params: 3,841 (15.00 KB)

 Non-trainable params: 0 (0.00 B)

In [10]:
# 8. 모델 학습
history = model.fit(X_train, y_train, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 1709365.2500 - mae: 1232.3420 - val_loss: 1812201.0000 - val_mae: 1267.0894
Epoch 2/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1691373.7500 - mae: 1225.1024 - val_loss: 1776184.7500 - val_mae: 1252.7546
Epoch 3/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1620210.2500 - mae: 1195.8008 - val_loss: 1653125.7500 - val_mae: 1202.3783
Epoch 4/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1416228.0000 - mae: 1106.7542 - val_loss: 1347046.2500 - val_mae: 1066.3129
Epoch 5/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1003669.0000 - mae: 899.4700 - val_loss: 836966.5000 - val_mae: 791.2966
Epoch 6/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 504576.0625 - mae: 583.8771 - val_loss: 385815.4375 - val_mae: 507.3466
Epoch 7/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 245744.2969 - mae: 411.4619 - val_loss: 260008.5000 - val_mae: 432.3715
Epoch 8/50
40/40 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 203902.4062 - m

In [11]:
# 9. 테스트 데이터 예측
# 회귀 모델의 predict()는 연속된 숫자를 반환하므로 .flatten()으로 1차원 변환
y_pred = model.predict(X_test).flatten()

13/13 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step


In [12]:
# 10. 성능 평가
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- 회귀 성능 평가 결과 ---")
print(f"MAE (평균 절대 오차) : {mae:.4f}")
print(f"MSE (평균 제곱 오차) : {mse:.4f}")
print(f"R² Score (결정계수) : {r2:.4f}")


--- 회귀 성능 평가 결과 ---
MAE (평균 절대 오차) : 391.3065
MSE (평균 제곱 오차) : 206571.1875
R² Score (결정계수) : -0.0966
